In [ ]:
import sys
sys.path.insert(0, "/workspace/src")

import zarr
from datasets.cwb import _ZarrDataset

DATA_PATH = "/workspace/data/cwa_dataset_storm.zarr"

# zarr v3 requires consolidated metadata; write it if missing
zarr.consolidate_metadata(DATA_PATH)

dataset = _ZarrDataset(DATA_PATH)
print(f"Dataset length: {len(dataset)}")


In [ ]:
# Input (ERA5) and output (CWB) channels
input_channels = dataset.input_channels()
output_channels = dataset.output_channels()

print(f"Input channels  ({len(input_channels)}):")
for ch in input_channels:
    print(f"  {ch.name:<40} level={ch.level}")

print(f"\nOutput channels ({len(output_channels)}):")
for ch in output_channels:
    print(f"  {ch.name:<40} level={ch.level}")


In [ ]:
import numpy as np

# Normalization stats (center/scale stored inside the zarr group)
era5_center = np.asarray(dataset.group["era5_center"])
era5_scale  = np.asarray(dataset.group["era5_scale"])
cwb_center  = np.asarray(dataset.group["cwb_center"])
cwb_scale   = np.asarray(dataset.group["cwb_scale"])

print(f"{'Input variable':<40} {'Center':>10} {'Scale':>10}")
print("-" * 62)
for ch, c, s in zip(input_channels, era5_center, era5_scale):
    print(f"  {ch.name:<38} {c:10.4f} {s:10.4f}")

print(f"\n{'Output variable':<40} {'Center':>10} {'Scale':>10}")
print("-" * 62)
for ch, c, s in zip(output_channels, cwb_center, cwb_scale):
    print(f"  {ch.name:<38} {c:10.4f} {s:10.4f}")


In [ ]:
# Sample shape and time range
y, x = dataset[0]
print(f"Input tensor  shape: {tuple(x.shape)}  (channels={x.shape[0]}, H={x.shape[1]}, W={x.shape[2]})")
print(f"Target tensor shape: {tuple(y.shape)}  (channels={y.shape[0]}, H={y.shape[1]}, W={y.shape[2]})")
print(f"\nTime range: {dataset.time()[0]}  →  {dataset.time()[-1]}")


In [ ]:
import matplotlib.pyplot as plt


def plot_variable(dataset, source, channel_idx, sample_idx=0, cmap="RdBu_r"):
    """Plot a single channel from 'era5' (input) or 'cwb' (output)."""
    arr = np.asarray(dataset.group[source][sample_idx, channel_idx])
    channels = dataset.input_channels() if source == "era5" else dataset.output_channels()
    ch = channels[channel_idx]
    label = f"{ch.name} {ch.level}".strip()

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(arr, cmap=cmap, origin="upper")
    plt.colorbar(im, ax=ax, label=label)
    ax.set_title(f"{source}  [{label}]  —  sample {sample_idx}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.tight_layout()
    plt.show()


In [ ]:
plot_variable(dataset, "era5", channel_idx=17, sample_idx=0)

In [ ]:
plot_variable(dataset, "cwb", channel_idx=1, sample_idx=0)

In [ ]:
import numpy as np
import matplotlib.animation as animation
from IPython.display import HTML

plt.rcParams["animation.embed_limit"] = 100  # MB


def animate_variable(dataset, source, channel_idx, cmap="RdBu_r", interval=100,
                     start=0, n_steps=None):
    """Animate a channel over a contiguous range of valid samples.

    Args:
        source: "era5" or "cwb"
        channel_idx: index into that source's channel list
        interval: milliseconds between frames
        start: first valid sample index to include
        n_steps: number of consecutive frames to animate (None = all remaining)
    """
    channels = dataset.input_channels() if source == "era5" else dataset.output_channels()
    ch = channels[channel_idx]
    label = f"{ch.name} {ch.level}".strip()

    # Index only valid rows so frames and times stay aligned
    valid_idxs = np.where(dataset.valid_times)[0]
    end = start + n_steps if n_steps is not None else len(valid_idxs)
    valid_idxs = valid_idxs[start:end]

    all_frames = np.stack(
        [np.asarray(dataset.group[source][int(i), channel_idx]) for i in valid_idxs]
    )
    times = list(dataset._read_time()[valid_idxs])
    vmin, vmax = all_frames.min(), all_frames.max()

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(all_frames[0], cmap=cmap, origin="upper", vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label=label)
    title = ax.set_title("")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.tight_layout()

    def update(i):
        im.set_data(all_frames[i])
        title.set_text(f"{source}  [{label}]  —  {times[i]}")
        return im, title

    anim = animation.FuncAnimation(
        fig, update, frames=len(all_frames), interval=interval, blit=True
    )
    plt.close(fig)

    # To save as GIF instead (Pillow is already installed):
    # anim.save("animation.gif", writer="pillow", fps=10)

    return HTML(anim.to_jshtml())


In [ ]:
animate_variable(dataset, "cwb", channel_idx=1, start=48, n_steps=48)
